In [ ]:
import io

from PIL import Image, ImageFilter, ImageOps
import matplotlib.pyplot as plt
import pytesseract
from pytesseract import Output

In [ ]:
OCR_LANG = "vie+eng"
OCR_RENDER_DPI = 300
OCR_LAYOUT_PSM = 3
OCR_MIN_CONFIDENCE = 30
OCR_MIN_TEXT_LENGTH = 40

TESSERACT_CMD = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def configure_tesseract(cmd: Optional[str] = None):
    if cmd:
        pytesseract.pytesseract.tesseract_cmd = cmd

In [ ]:
def build_empty_sections()-> dict[str, list[str]]:
    return {section: [] for section in PREFERRED_SECTION_ORDER}

def build_extraction_result(pdf_type: str,
    method: str,
    text: str = "",
    warnings: list[str] | None = None,
) -> dict[str, Any]:
    return {
        "text": text,
        "sections": build_empty_sections(),
        "pdf_type": pdf_type,
        "method": method,
        "warnings": list(warnings or []),
        "layout_debug": [],
    }

Hàm chuyển 1 trang pdf sang ảnh nhận đầu vào là 1 trang pdf và độ phân giải:
- 1 trang pdf mặc định có độ phân giải 72 DPI, nên để đạt được độ phân giải mong muốn, ta cần zoom lên theo tỷ lệ dpi/72
- Tạo ma trận hình học để phóng to pdf theo X và Y đúng tỉ lệ vừa tính (gấp dpi/72)
- Gói dữ liệu pixel thpp thành 1 chuỗi byte theo chuẩn cấu trúc của png
- Chuyển đổi sang RGB để đảm bảo tesseract xử lý tốt hơn

In [ ]:
def render_pdf_page_to_image(page: fitz.Page,
    dpi: int = OCR_RENDER_DPI, #dpi: độ phân giải
) -> Image.Image:
    zoom = dpi/72 
    matrix = fitz.Matrix(zoom, zoom) 
    pixmap = page.get_pixmap(matrix=matrix, alpha=False)
    image_bytes = pixmap.tobytes("png")
    return Image.open(io.BytesIO(image_bytes)).convert("RGB") 

Hàm tiền xử lý ảnh trước OCR (nhận tham số là 1 ảnh và trả về 1 ảnh đã xử lý):
- Chuyển ảnh sang thang độ xám (vì đôi khi chữ sẽ có những màu khác nhau, đưa hết về màu xám để loại bỏ thông tin màu không cần thiết)
- Tự động điều chỉnh độ tương phản để làm rõ chữ
- Áp dụng bộ lọc trung vị để giảm nhiễu (trượt qua toàn bộ ảnh, thay thế giá trị của điểm ảnh ở giữa bằng giá trị trung bình của các điểm xung quanh => Xóa các hạt nhiễu li ti trên ảnh)
- Chuyển ảnh thành đen trắng với ngưỡng 180
    + Nếu độ sáng của pixel > 180 => ép nó thành màu trắng (255)
    + Nếu độ sáng < = 180 => Ép nó thành màu đen (0)
- Chuyển ảnh nhị phân sang thang độ xám để OCR hoạt động tốt hơn

In [ ]:
def show_image_for_debug(
    image: Image.Image,
    title: str = "OCR Debug Image",
    figsize: tuple[int, int] = (10, 12),
) -> None:
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

def preprocess_image_for_ocr(image: Image.Image, show_steps: bool = False) -> Image.Image:
    gray = ImageOps.grayscale(image) 
    gray = ImageOps.autocontrast(gray) 
    denoised = gray.filter(ImageFilter.MedianFilter(size=3)) 
    
    binary = denoised.point(lambda x: 255 if x > 180 else 0, mode="1")
    processed = binary.convert("L")

    if show_steps:
        show_image_for_debug(gray, title="1. Gray + Autocontrast")
        show_image_for_debug(denoised, title="2. Denoised")
        show_image_for_debug(processed, title="3. Final Preprocessed Image")

    return processed


Tách từ một kèm theo tọa độ chính xác của nó thông qua ảnh trong pdf:
- Đưa ảnh qua tiền xử lý
- Trích xuất dữ liệu chi tiết, dùng image_to_data
    + Sử dụng tham số output_type = Output.DICT => Yêu cầu Tesseract trả kết quả về dạng một Dictionary cuả Python
- Tạo valid_words chứa kết quả
- Vòng lặp chạy qua từng từ mà ocr nhận diện được
    + Nó lấy ra chữ tại vị trí index, loại bỏ khoảng trắng thừa và đưa qua hàm clean_line_texxt để xóa bỏ kí tự rác
- conf: thể hiện mức độ tự tin của engine khi đọc từ đó
- Nếu như điểm tự tin nhỏ hơn ngưỡng cho phép thì không lưu từ đó
- Trích xuất tọa độ hộp bao quanh từ đó
=> Đóng gói kết quả và lưu vào valid_words

In [ ]:
def extract_words_from_ocr_image(image: Image.Image, lang: str = OCR_LANG, psm: int = OCR_LAYOUT_PSM, min_confidence: int = OCR_MIN_CONFIDENCE, show_preprocessed: bool = False) -> list[dict[str, Any]]:
    configure_tesseract() 

    processed_image = preprocess_image_for_ocr(
        image = image,
        show_steps=show_preprocessed
    )

    ocr_data = pytesseract.image_to_data(
        processed_image,
        lang=lang,
        config=f"--oem 3 --psm {psm}",
        output_type=Output.DICT
    )

    valid_words: list[dict[str, Any]] = []
    total_items = len(ocr_data["text"])

    for index in range(total_items):
        text = str(ocr_data["text"][index] or "").strip()
        text = clean_line_text(text)

        conf_raw = str(ocr_data["conf"][index]).strip()

        try:
            confidence = float(conf_raw)
        except ValueError:
            confidence = -1.0

        if not text:
            continue
        if confidence < min_confidence:
            continue

        left = float(ocr_data["left"][index])
        top = float(ocr_data["top"][index])
        width = float(ocr_data["width"][index])
        height = float(ocr_data["height"][index])

        x0 = left
        y0 = top
        x1 = left + width
        y1 = top + height
        
        valid_words.append(
            {
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text,
                "y_center": (y0 + y1) / 2,
                "confidence": confidence,
            }
        )

    return valid_words

In [ ]:
def append_layout_result_from_words(
    result: dict[str, Any],
    words: list[dict[str, Any]],
    page_width: float,
    page_index: int,
    y_tolerance: float = 4.0,
    x_gap_threshold: float = 50.0,
    column_margin: float = 8.0,
) -> None:
    if not words:
        result["warnings"].append(f"Trang {page_index + 1} không lấy được word nào.")
        return

    rows = group_words_into_rows(words, y_tolerance=y_tolerance)
    rough_lines = rows_to_lines_by_x_gap(rows, x_gap_threshold=x_gap_threshold)
    layout = detect_two_column_layout(rough_lines, page_width=page_width)

    if layout:
        left_lines, right_lines = rows_to_column_lines(
            rows,
            right_start_x=layout["right_start"],
            column_margin=column_margin,
        )

        left_leading, left_sections = parse_sections_from_lines(left_lines)
        right_leading, right_sections = parse_sections_from_lines(right_lines)

        if page_index == 0:
            result["sections"]["header"].extend(left_leading + right_leading)
        else:
            result["sections"]["other"].extend(left_leading + right_leading)

        merge_sections(result["sections"], left_sections)
        merge_sections(result["sections"], right_sections)

        result["layout_debug"].append(
            {
                "page": page_index + 1,
                "layout": "two_columns_from_ocr_words",
                "left_start": layout["left_start"],
                "right_start": layout["right_start"],
                "split_x": layout["split_x"],
                "left_line_count": len(left_lines),
                "right_line_count": len(right_lines),
                "word_count": len(words),
            }
        )
        return

    lines = rows_to_single_column_lines(rows)
    leading_lines, sections = parse_sections_from_lines(lines)

    if page_index == 0:
        result["sections"]["header"].extend(leading_lines)
    else:
        result["sections"]["other"].extend(leading_lines)

    merge_sections(result["sections"], sections)

    result["layout_debug"].append(
        {
            "page": page_index + 1,
            "layout": "single_column_from_ocr_words",
            "line_count": len(lines),
            "word_count": len(words),
        }
    )
    
    